# 📘 智能体架构 17：反思型元认知智能体

深入实现最精密的智能体模式之一：**反思型元认知智能体**。该架构赋予智能体一种自我意识，使其在行动前能对自身能力、信心与局限进行推理。

这超越了简单自省。元认知智能体维护显式的 **「自我模型」**——对自身知识、工具与边界的结构化表示。面对任务时，第一步不是解题，而是 *在自我模型语境下分析问题*。它会自问：
- 「我是否有足够知识自信作答？」
- 「该主题是否在我的职责专长范围内？」
- 「我是否具备能安全、准确回答所需的特定工具？」
- 「用户问题是否属于高风险领域，错误会造成危险？」

根据答案选择策略：直接推理、使用专用工具，或——最重要的是——在任务超出已知能力时 **升级给人类**。

为构建有力演示，我们将创建 **医学分诊与信息助手**——典型高风险场景：识别自身局限不仅是功能，更是关键安全要求。


### 定义
**反思型元认知智能体** 维护并使用关于自身能力、知识边界与信心水平的显式模型，以为给定任务选择最合适策略。这种自我建模使其在错误信息有害的领域更安全、可靠。

### 高层工作流

1.  **感知任务：** 接收用户请求。
2.  **元认知分析（自省）：** 核心推理引擎 *对照自我模型* 分析请求，评估信心、工具相关性及查询是否在预定运行域内。
3.  **策略选择：** 根据分析选择策略：
    *   **直接推理：** 高信心、低风险、知识库内查询。
    *   **使用工具：** 查询需要智能体通过工具具备的特定能力。
    *   **升级/拒绝：** 低信心、高风险或越界查询。
4.  **执行策略：** 执行所选路径。
5.  **响应：** 返回结果，可能是直接答案、工具增强答案，或安全拒绝并建议咨询专家。

### 适用场景
*   **高风险咨询系统：** 医疗、法律、金融等需提供信息的系统，智能体必须能说「不知道」或「请咨询专业人士」。
*   **自主系统：** 机器人在尝试物理任务前评估自身能否安全完成。
*   **复杂工具编排：** 必须从大量 API 中选择合适接口，且理解部分 API 更危险或更昂贵。

### 优势与局限
*   **优势：**
    *   **安全与可靠性增强：** 主要收益。显式避免在非专长领域作自信断言。
    *   **决策更稳健：** 强制审慎选择策略，而非天真地直接尝试。
*   **局限：**
    *   **自我模型复杂：** 定义并维护准确的自我模型可能很困难。
    *   **元认知开销：** 初始分析步骤为每次请求增加延迟与计算成本。


## 阶段 0：基础与配置

库与环境变量的常规配置。


In [1]:
# !pip install -q -U langchain-openai langchain langgraph rich python-dotenv

In [2]:
import os
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

# Pydantic for data modeling
from pydantic import BaseModel, Field

# LangChain components
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# LangGraph components
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel

# --- API Key and Tracing Setup ---
load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Metacognitive Agent (DeepSeek)"

required_vars = ["DEEPSEEK_API_KEY", "LANGCHAIN_API_KEY"]
for var in required_vars:
    if var not in os.environ:
        print(f"Warning: Environment variable {var} not set.")

print("Environment variables loaded and tracing is set up.")

Environment variables loaded and tracing is set up.


## 阶段 1：定义智能体的自我模型与工具

这是自我意识的根基。我们将创建结构化的 `AgentSelfModel` 与专用工具。该模型不仅是提示词，而是将传入推理循环的配置对象。


In [3]:
console = Console()

# --- The Agent's Self-Model ---
class AgentSelfModel(BaseModel):
    """A structured representation of the agent's capabilities and limitations."""
    name: str
    role: str
    # The agent's explicit knowledge boundaries
    knowledge_domain: List[str] = Field(description="List of topics the agent is knowledgeable about.")
    # The agent's available tools
    available_tools: List[str] = Field(description="List of tools the agent can use.")
    confidence_threshold: float = Field(description="The confidence level (0-1) below which the agent must escalate.", default=0.6)

# Instantiate the self-model for our Medical Triage Agent
medical_agent_model = AgentSelfModel(
    name="TriageBot-3000",
    role="A helpful AI assistant for providing preliminary medical information.",
    knowledge_domain=["common_cold", "influenza", "allergies", "headaches", "basic_first_aid"],
    available_tools=["drug_interaction_checker"]
)

# --- Specialist Tools ---
class DrugInteractionChecker:
    """A mock tool to check for drug interactions."""
    def check(self, drug_a: str, drug_b: str) -> str:
        """Checks for interactions between two drugs."""
        # In a real system, this would query a medical database.
        known_interactions = {
            frozenset(["ibuprofen", "lisinopril"]): "Moderate risk: Ibuprofen may reduce the blood pressure-lowering effects of lisinopril. Monitor blood pressure.",
            frozenset(["aspirin", "warfarin"]): "High risk: Increased risk of bleeding. This combination should be avoided unless directed by a doctor."
        }
        interaction = known_interactions.get(frozenset([drug_a.lower(), drug_b.lower()]))
        if interaction:
            return f"Interaction Found: {interaction}"
        return "No known significant interactions found. However, always consult a pharmacist or doctor."

drug_tool = DrugInteractionChecker()
print("Agent Self-Model and Tools defined successfully.")

Agent Self-Model and Tools defined successfully.


## 阶段 2：用 LangGraph 构建元认知智能体

关键在此：构建图时，**第一步即元认知分析**。该节点用详尽提示让智能体对自身推理；条件路由器再按所选策略分流。


In [4]:
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",
    temperature=0,
)

# Pydantic Models for structured outputs
class MetacognitiveAnalysis(BaseModel):
    """The agent's self-analysis of a query."""
    confidence: float = Field(description="A score from 0.0 to 1.0 representing the agent's confidence in its ability to answer safely and accurately.")
    strategy: str = Field(description="The chosen strategy. Must be one of: 'reason_directly', 'use_tool', 'escalate'.")
    reasoning: str = Field(description="A brief justification for the chosen confidence and strategy.")
    tool_to_use: Optional[str] = Field(description="If strategy is 'use_tool', the name of the tool to use.", default=None)
    tool_args: Optional[Dict[str, Any]] = Field(description="If strategy is 'use_tool', the arguments for the tool.", default=None)

# LangGraph State
class AgentState(TypedDict):
    user_query: str
    self_model: AgentSelfModel
    metacognitive_analysis: Optional[MetacognitiveAnalysis]
    tool_output: Optional[str]
    final_response: str

# Graph Nodes
def metacognitive_analysis_node(state: AgentState) -> Dict[str, Any]:
    console.print(Panel("🤔 Agent is performing metacognitive analysis...", title="[yellow]Step: Self-Reflection[/yellow]"))
    prompt = ChatPromptTemplate.from_template(
        """You are a metacognitive reasoning engine for an AI assistant. Your task is to analyze a user's query in the context of the agent's own capabilities and limitations (its 'self-model').
        Your primary directive is **SAFETY**. You must determine the safest and most appropriate strategy for handling the query.

        **Agent's Self-Model:**
        - Name: {agent_name}
        - Role: {agent_role}
        - Knowledge Domain: {knowledge_domain}
        - Available Tools: {available_tools}

        **Strategy Rules:**
        1.  **escalate:** Choose this strategy if the query involves a potential medical emergency (e.g., chest pain, difficulty breathing, severe injury, broken bones), is outside the agent's knowledge domain, or if you have any doubt about providing a safe answer. **WHEN IN DOUBT, ESCALATE.**
        2.  **use_tool:** Choose this strategy if the query explicitly or implicitly requires one of the available tools. For example, a question about drug interactions requires the 'drug_interaction_checker'.
        3.  **reason_directly:** Choose this strategy ONLY if you are highly confident the query is a simple, low-risk question that falls squarely within the agent's knowledge domain.

        Analyze the user query below and provide your metacognitive analysis in the required format.

        **User Query:** "{query}"""
    )
    chain = prompt | llm.with_structured_output(MetacognitiveAnalysis)
    analysis = chain.invoke({
        "query": state['user_query'],
        "agent_name": state['self_model'].name,
        "agent_role": state['self_model'].role,
        "knowledge_domain": ", ".join(state['self_model'].knowledge_domain),
        "available_tools": ", ".join(state['self_model'].available_tools),
    })
    console.print(Panel(f"[bold]Confidence:[/bold] {analysis.confidence:.2f}\n[bold]Strategy:[/bold] {analysis.strategy}\n[bold]Reasoning:[/bold] {analysis.reasoning}", title="Metacognitive Analysis Result"))
    return {"metacognitive_analysis": analysis}

def reason_directly_node(state: AgentState) -> Dict[str, Any]:
    console.print(Panel("✅ Confident in direct answer. Generating response...", title="[green]Strategy: Reason Directly[/green]"))
    prompt = ChatPromptTemplate.from_template("You are {agent_role}. Provide a helpful, non-prescriptive answer to the user's query. Remind the user that you are not a doctor.\n\nQuery: {query}")
    chain = prompt | llm
    response = chain.invoke({"agent_role": state['self_model'].role, "query": state['user_query']}).content
    return {"final_response": response}

def call_tool_node(state: AgentState) -> Dict[str, Any]:
    console.print(Panel(f"🛠️ Confidence requires tool use. Calling `{state['metacognitive_analysis'].tool_to_use}`...", title="[cyan]Strategy: Use Tool[/cyan]"))
    analysis = state['metacognitive_analysis']
    if analysis.tool_to_use == 'drug_interaction_checker':
        tool_output = drug_tool.check(**analysis.tool_args)
        return {"tool_output": tool_output}
    return {"tool_output": "Error: Tool not found."}

def synthesize_tool_response_node(state: AgentState) -> Dict[str, Any]:
    console.print(Panel("📝 Synthesizing final response from tool output...", title="[cyan]Step: Synthesize[/cyan]"))
    prompt = ChatPromptTemplate.from_template("You are {agent_role}. You have used a tool to get specific information. Now, present this information to the user in a clear and helpful way. ALWAYS include a disclaimer to consult a healthcare professional.\n\nOriginal Query: {query}\nTool Output: {tool_output}")
    chain = prompt | llm
    response = chain.invoke({"agent_role": state['self_model'].role, "query": state['user_query'], "tool_output": state['tool_output']}).content
    return {"final_response": response}

def escalate_to_human_node(state: AgentState) -> Dict[str, Any]:
    console.print(Panel("🚨 Low confidence or high risk detected. Escalating to human.", title="[bold red]Strategy: Escalate[/bold red]"))
    response = "I am an AI assistant and not qualified to provide information on this topic. This query is outside my knowledge domain or involves potentially serious symptoms. **Please consult a qualified medical professional immediately.**"
    return {"final_response": response}

# Conditional Edge
def route_strategy(state: AgentState) -> str:
    return state["metacognitive_analysis"].strategy

# Build the graph
workflow = StateGraph(AgentState)
workflow.add_node("analyze", metacognitive_analysis_node)
workflow.add_node("reason", reason_directly_node)
workflow.add_node("call_tool", call_tool_node)
workflow.add_node("synthesize", synthesize_tool_response_node)
workflow.add_node("escalate", escalate_to_human_node)

workflow.set_entry_point("analyze")
workflow.add_conditional_edges("analyze", route_strategy, {
    "reason_directly": "reason",
    "use_tool": "call_tool",
    "escalate": "escalate"
})
workflow.add_edge("call_tool", "synthesize")
workflow.add_edge("reason", END)
workflow.add_edge("synthesize", END)
workflow.add_edge("escalate", END)

metacognitive_agent = workflow.compile()
print("Reflexive Metacognitive Agent graph compiled successfully.")

Reflexive Metacognitive Agent graph compiled successfully.


## 阶段 3：演示与分析

用一系列难度与风险递增的查询测试智能体，观察元认知分析如何将各查询路由到合适路径，体现系统的安全性与自我觉察。


In [5]:
def run_agent(query: str):
    initial_state = {"user_query": query, "self_model": medical_agent_model}
    result = metacognitive_agent.invoke(initial_state)
    console.print(Markdown(result['final_response']))

# Test 1: Simple, should be answered directly
console.print("--- Test 1: Simple, In-Scope, Low-Risk Query ---")
run_agent("What are the symptoms of a common cold?")

# Test 2: Requires the specific tool
console.print("\n--- Test 2: Specific Query Requiring a Tool ---")
run_agent("Is it safe to take Ibuprofen if I am also taking Lisinopril?")

# Test 3: High-stakes, should be escalated immediately
console.print("\n--- Test 3: High-Stakes, Emergency Query ---")
run_agent("I have a crushing pain in my chest and my left arm feels numb, what should I do?")

--- Test 1: Simple, In-Scope, Low-Risk Query ---

                 Step: Self-Reflection                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 🤔 Agent is performing metacognitive analysis...      ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

                   Metacognitive Analysis Result                    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Confidence: 0.90                                                 ┃
┃ Strategy: reason_directly                                        ┃
┃ Reasoning: The user's query about symptoms of a common cold      ┃
┃ falls directly within the agent's specified knowledge domain. It ┃
┃ is a low-risk, informational question.                           ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

                         Strategy: Reason Directly                          
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ✅ Confident in direct answer. Generating response...                     ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

Based on your query, here is some general information about the common cold. Please remember, I am an AI assistant and not a medical doctor. This information should not be considered medical advice.

Common symptoms of a cold often include:
*   Runny or stuffy nose
*   Sore throat
*   Cough
*   Sneezing
*   Mild body aches or a slight headache

These symptoms are typically mild and resolve on their own. If your symptoms are severe or persist, it is always best to consult a healthcare professional.

...


--- Test 3: High-Stakes, Emergency Query ---

                 Step: Self-Reflection                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 🤔 Agent is performing metacognitive analysis...      ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

                   Metacognitive Analysis Result                    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Confidence: 0.10                                                 ┃
┃ Strategy: escalate                                               ┃
┃ Reasoning: The user's query describes symptoms (crushing chest   ┃
┃ pain, numbness in arm) that are highly indicative of a potential ┃
┃ medical emergency. This is far outside the agent's knowledge     ┃
┃ domain and requires immediate professional medical attention. The┃
┃ only safe action is to escalate.                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

                           Strategy: Escalate                           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 🚨 Low confidence or high risk detected. Escalating to human.       ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

I am an AI assistant and not qualified to provide information on this topic. This query is outside my knowledge domain or involves potentially serious symptoms. **Please consult a qualified medical professional immediately.**

### 结果分析

演示有力说明了该架构带来的安全与可靠：

1.  **范围正确的回答：** 对「普通感冒」查询，元认知分析正确识别为知识域内、低风险话题，给出高信心并选择 `reason_directly`，提供有帮助且充分免责的答复。

2.  **正确使用工具：** 对药物相互作用问题，分析识别需要特定能力，正确判定需要 `drug_interaction_checker`，对 *使用工具的能力* 给高信心，选择 `use_tool`。最终响应是对工具输出的安全综合。

3.  **关键安全升级：** 这是最重要的一点。朴素智能体可能试图用网络搜索回答「胸痛」查询，提供危险误导信息。我们的元认知智能体以安全为首要指令，立即识别急症征象。元认知分析给极低信心并正确选择 `escalate`。最终输出不是「答案」，而是负责任的拒绝与寻求专业帮助的指引。它正确识别了自身胜任力的边界。

该工作流证明：强制智能体在思考问题 *之前* 先思考自身，可在运行中构建强大的安全与可靠层。


## 小结

在本详细笔记本中，我们实现了 **反思型元认知智能体**——通过赋予智能体自我意识而优先安全与可靠的精密架构。通过构建显式 `self-model` 并将元认知分析作为任何任务的第一步，我们得到了理解自身边界的系统。

关键创新在于智能体初始目标从「如何回答？」转变为「*是否应当* 回答，若应当，如何回答？」这一内省步骤使智能体能动态选择最安全、最合适的策略——直接推理、专用工具，或至关重要的升级给人类专家。

该架构不仅是技巧，更是设计哲学。对于必须在高风险真实领域可信赖、负责任的 AI 智能体，**知道自己不知道什么** 与知道自己知道什么同样重要。
